# Bell Pepper Dataset generator

In [1]:
# imports

import os
from IPython.display import display
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig, pipeline
import gradio as gr
import torch

from styles import CSS
import csv
import io

In [2]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

hf_token = os.getenv('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
system_message = """
You are generating synthetic agricultural data for a bell pepper farm.

Generate a CSV dataset with the following columns in this exact order:

Seed_Variant,Disease_Resistance,Fertilizer_Used,Planting_Date,Harvest_Date,Irrigation_Method,Yield_kg

Rules:
- Output ONLY valid CSV.
- Do NOT include explanations, comments, or markdown.
- Do NOT wrap the output in code blocks.
- The first line must be the header.
- Generate 30 rows of realistic farm data.
- Dates must be in YYYY-MM-DD format.
- Use commas as separators.
- Do not include empty rows.

Return ONLY the CSV content.
"""

user_prompt = f"""
I need you to generate a dataset for a bell pepper farm. I need it to make for future reference as well as for training a model.
"""


In [4]:
MODEL = 'Qwen/Qwen2.5-3B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL).to(device)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [5]:
def to_csv(csv_text):
    reader = csv.reader(io.StringIO(csv_text))

    with open("output.csv", "w", newline="") as f:
        writer = csv.writer(f)
        for row in reader:
            writer.writerow(row)

In [8]:
def generate(quant=True, max_new_tokens=800):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt}
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True
    ).to(device)

    attention_mask = torch.ones_like(input_ids, dtype=torch.long, device=device)

    streamer = TextStreamer(tokenizer)

    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        streamer=streamer
    )

    # remove prompt tokens
    res = tokenizer.decode(
        outputs[0][input_ids.shape[-1]:],
        skip_special_tokens=True
    )

    to_csv(res)

    return res

In [ ]:

with gr.Blocks() as ui:
    
    output_box = gr.Textbox(
        label="Output",
        lines=15
    )

    generate_btn = gr.Button("Generate")

    generate_btn.click(
        fn=generate,
        inputs=[],
        outputs=output_box
    )

ui.launch()


* Running on local URL:  http://127.0.0.1:7904
* To create a public link, set `share=True` in `launch()`.


<|im_start|>system

You are generating synthetic agricultural data for a bell pepper farm.

Generate a CSV dataset with the following columns in this exact order:

Seed_Variant,Disease_Resistance,Fertilizer_Used,Planting_Date,Harvest_Date,Irrigation_Method,Yield_kg

Rules:
- Output ONLY valid CSV.
- Do NOT include explanations, comments, or markdown.
- Do NOT wrap the output in code blocks.
- The first line must be the header.
- Generate 30 rows of realistic farm data.
- Dates must be in YYYY-MM-DD format.
- Use commas as separators.
- Do not include empty rows.

Return ONLY the CSV content.
<|im_end|>
<|im_start|>user

I need you to generate a dataset for a bell pepper farm. I need it to make for future reference as well as for training a model.
<|im_end|>
<|im_start|>assistant
Variant2,Moderate,Chemical,2023-10-10,2024-03-05,Well_water,82
Seed_Variant,Disease_Resistance,Fertilizer_Used,Planting_Date,Harvest_Date,Irrigation_Method,Yield_kg
Variant3,Low,Organic,2023-10-20,2024-02-28,Sp